# 12 - Off-Exchange and TRF Reconciliation (Granular)

## Objetivo cientifico
Estimar participaci?n off-exchange/TRF y detectar concentraciones an?malas por segmento temporal.

## Riesgo metodologico clave
Si `venue/exchange` viene codificado sin mapeo confiable a TRF/off-exchange, una estimaci?n de share=0 puede ser falsa.

## Politica operativa
- Si no hay columna de venue: `NOT_APPLICABLE`.
- Si hay venue pero no es clasificable con confianza: `NOT_APPLICABLE` (o `WARN` de baja confianza).
- Solo con clasificaci?n defendible se emite PASS/WARN/FAIL cuantitativo.

Referencias:
- FINRA TRF overview: https://www.finra.org/filing-reporting/trade-reporting-facility-trf
- FINRA OTC transparency: https://www.finra.org/filing-reporting/otc-transparency
- FINRA short sale volume: https://www.finra.org/finra-data/browse-catalog/short-sale-volume



## Paso 1 - Setup y muestra controlada


In [1]:
NOTEBOOK_ID = "12_off_exchange_trf_reconciliation"

import json
import os
import sys
import uuid
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import polars as pl

try:
    import matplotlib.pyplot as plt
except Exception as e:
    plt = None
    print("matplotlib not available:", e)


def detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd] + list(cwd.parents)
    for cand in candidates:
        if (cand / "data" / "manifests").exists() and (cand / "notebooks" / "01_data_integrity").exists():
            return cand
    return cwd

PROJECT_ROOT = detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

DATA_ROOT = Path(os.getenv("DATA_CACHE_DIR", r"C:\TSIS_Data\data")).resolve()
SYMBOLS = ["AABA"]
MAX_FILES = 300
ROWS_PER_FILE = 2000

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
OUT_DIR = PROJECT_ROOT / "runs" / "data_quality" / NOTEBOOK_ID / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT).decode().strip()
except Exception:
    git_commit = "<unknown>"

print("Run ID:", RUN_ID)
print("Out dir:", OUT_DIR)
print("Data root:", DATA_ROOT)
print("Symbols:", SYMBOLS)


def list_trade_files(data_root: Path, symbols: list[str], max_files: int) -> list[Path]:
    files = []
    roots = ["trades_ticks_2004_2018", "trades_ticks_2019_2025"]
    for r in roots:
        for s in symbols:
            base = data_root / r / s
            if not base.exists():
                continue
            for fp in base.rglob("*.parquet"):
                files.append(fp)
                if len(files) >= max_files:
                    return files
    return files


def sample_concat(files: list[Path], rows_per_file: int) -> pl.DataFrame:
    parts = []
    for fp in files:
        try:
            df = pl.read_parquet(fp).head(rows_per_file)
            parts.append(df)
        except Exception:
            continue
    if not parts:
        return pl.DataFrame()
    return pl.concat(parts, how="diagonal_relaxed")



Run ID: 20260213_183454_19355fad
Out dir: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\12_off_exchange_trf_reconciliation\20260213_183454_19355fad
Data root: C:\TSIS_Data\data
Symbols: ['AABA']


**Interpretacion de salida anterior**

Confirma trazabilidad del run y alcance de muestra.


## Paso 2A - Schema gate y detectabilidad de venue


In [2]:
trade_files = list_trade_files(DATA_ROOT, SYMBOLS, MAX_FILES)
trades = sample_concat(trade_files, ROWS_PER_FILE)

venue_col = next((c for c in ["exchange", "exchange_id", "venue", "trf", "participant_id"] if c in trades.columns), None)
size_col = next((c for c in ["size", "trade_size", "s", "volume"] if c in trades.columns), None)

print("rows:", trades.height)
print("columns:", sorted(trades.columns))
print("venue_col:", venue_col, "size_col:", size_col)

if trades.height == 0:
    schema_gate = pl.DataFrame([{
        "rows_sampled": 0,
        "trade_files_sampled": int(len(trade_files)),
        "venue_col": venue_col,
        "size_col": size_col,
        "venue_detectability": "not_available",
        "off_exchange_check_status": "not_available",
    }])
else:
    if venue_col is None:
        detectability = "not_available"
        check_status = "not_available"
    else:
        vu = trades.select(pl.col(venue_col).cast(pl.Utf8, strict=False).str.to_uppercase().alias("venue_u"))
        non_null = vu.filter(pl.col("venue_u").is_not_null() & (pl.col("venue_u") != "")).height
        token_hit = vu.filter(
            pl.col("venue_u").str.contains("TRF|FINRA|OTC|ADF", literal=False)
        ).height

        # if venue exists but no token hit at all, detectability is low unless mapped externally
        if non_null == 0:
            detectability = "not_available"
            check_status = "not_available"
        elif token_hit == 0:
            detectability = "low_confidence_unmapped_codes"
            check_status = "not_applicable_without_mapping"
        else:
            detectability = "direct_token_detectable"
            check_status = "available"

    schema_gate = pl.DataFrame([{
        "rows_sampled": int(trades.height),
        "trade_files_sampled": int(len(trade_files)),
        "venue_col": venue_col,
        "size_col": size_col,
        "venue_detectability": detectability,
        "off_exchange_check_status": check_status,
    }])

schema_gate



rows: 298791
columns: ['conditions', 'date', 'exchange', 'price', 'size', 'ticker', 'timestamp']
venue_col: exchange size_col: size


rows_sampled,trade_files_sampled,venue_col,size_col,venue_detectability,off_exchange_check_status
i64,i64,str,str,str,str
298791,300,"""exchange""","""size""","""low_confidence_unmapped_codes""","""not_applicable_without_mapping"""


**Interpretacion de salida anterior**

Si `venue_detectability=low_confidence_unmapped_codes`, la reconciliaci?n off-exchange no es defendible sin mapeo de c?digos de venue.


## Paso 2B - Metricas globales (solo si aplicable)


In [3]:
offex = pl.DataFrame()
d = pl.DataFrame()

gate_row = schema_gate.to_dicts()[0] if schema_gate.height > 0 else {}
is_applicable = gate_row.get("off_exchange_check_status") == "available"

if trades.height == 0 or not is_applicable:
    offex = pl.DataFrame([{
        "n_rows": int(trades.height),
        "total_qty": None,
        "off_exchange_qty": None,
        "off_exchange_share": None,
        "locked_applicability": True,
    }])
else:
    d = trades
    d = d.with_columns(pl.col(venue_col).cast(pl.Utf8, strict=False).str.to_uppercase().alias("venue_u"))

    if size_col is None:
        d = d.with_columns(pl.lit(1.0).alias("qty"))
    else:
        d = d.with_columns(pl.col(size_col).cast(pl.Float64, strict=False).fill_null(0.0).alias("qty"))

    d = d.with_columns(
        pl.col("venue_u").str.contains("TRF|FINRA|OTC|ADF", literal=False).alias("is_off_exchange")
    )

    total_qty = float(d.select(pl.col("qty").sum()).item()) if d.height > 0 else 0.0
    off_qty = float(d.filter(pl.col("is_off_exchange")).select(pl.col("qty").sum()).item()) if d.height > 0 else 0.0

    offex = pl.DataFrame([{
        "n_rows": int(d.height),
        "total_qty": total_qty,
        "off_exchange_qty": off_qty,
        "off_exchange_share": (off_qty / total_qty) if total_qty > 0 else None,
        "locked_applicability": False,
    }])

offex



n_rows,total_qty,off_exchange_qty,off_exchange_share,locked_applicability
i64,null,null,null,bool
298791,null,null,null,true


**Interpretacion de salida anterior**

Si `locked_applicability=True`, los valores cuantitativos no deben usarse para decisiones de negocio.


## Paso 2C - Granularidad por archivo/dia/sesion y venue


In [4]:
def _path_meta(fp: Path) -> dict:
    rel = fp.relative_to(DATA_ROOT)
    parts = rel.parts
    dataset_root = parts[0] if len(parts) > 0 else "<unknown>"
    ticker = parts[1] if len(parts) > 1 else "<unknown>"
    day = None
    for part in parts:
        if part.startswith("day="):
            day = part.split("=", 1)[1]
            break
    return {
        "dataset_root": dataset_root,
        "ticker": ticker,
        "day": day,
        "session": fp.stem,
        "file_path": str(fp),
    }


gran_file = pl.DataFrame()
venue_profile = pl.DataFrame()

if is_applicable and venue_col is not None:
    rows=[]
    venue_rows=[]
    for fp in trade_files:
        try:
            cols=pl.read_parquet(fp, n_rows=1).columns
            if venue_col not in cols:
                continue
            use=[venue_col]
            if size_col is not None and size_col in cols:
                use.append(size_col)
            t=pl.read_parquet(fp, columns=use).head(ROWS_PER_FILE)
            if t.height==0:
                continue
            if size_col is None or size_col not in t.columns:
                t=t.with_columns(pl.lit(1.0).alias('qty'))
            else:
                t=t.with_columns(pl.col(size_col).cast(pl.Float64, strict=False).fill_null(0.0).alias('qty'))
            t=t.with_columns(pl.col(venue_col).cast(pl.Utf8, strict=False).str.to_uppercase().alias('venue_u'))
            t=t.with_columns(pl.col('venue_u').str.contains('TRF|FINRA|OTC|ADF', literal=False).alias('is_off_exchange'))

            tq=float(t.select(pl.col('qty').sum()).item()) if t.height>0 else 0.0
            oq=float(t.filter(pl.col('is_off_exchange')).select(pl.col('qty').sum()).item()) if t.height>0 else 0.0
            rows.append({**_path_meta(fp),'n_rows':int(t.height),'total_qty':tq,'off_exchange_qty':oq,'off_exchange_share':(oq/tq if tq>0 else None)})

            vp=t.group_by('venue_u').agg([pl.len().alias('n_rows'),pl.col('qty').sum().alias('qty_sum')]).sort('n_rows',descending=True).head(10)
            for r in vp.to_dicts():
                venue_rows.append({**_path_meta(fp), **r})
        except Exception:
            continue

    gran_file = pl.DataFrame(rows) if rows else pl.DataFrame()
    venue_profile = pl.DataFrame(venue_rows) if venue_rows else pl.DataFrame()

if gran_file.height>0:
    print('Top by off_exchange_share:')
    print(gran_file.select(['day','session','n_rows','off_exchange_share']).sort('off_exchange_share',descending=True).head(10))
else:
    print('No granular off-exchange rows (not applicable or no detected mapping).')

if venue_profile.height>0:
    print('\nTop venue tokens (sample):')
    print(venue_profile.select(['venue_u','n_rows']).group_by('venue_u').agg(pl.col('n_rows').sum().alias('n_rows')).sort('n_rows',descending=True).head(20))

gran_file



No granular off-exchange rows (not applicable or no detected mapping).


shape: (0, 0)
┌┐
╞╡
└┘

**Interpretacion de salida anterior**

Esta salida muestra si el share est? concentrado en d?as/sesiones espec?ficas y qu? c?digos de venue dominan la muestra.


## Paso 3 - Visuales


In [5]:
if plt is not None and offex.height > 0:
    row=offex.to_dicts()[0]
    if row.get('off_exchange_share') is not None:
        plt.style.use("seaborn-v0_8-darkgrid")
        plt.figure(figsize=(6,4))
        plt.bar(['off_exchange','on_exchange'], [row['off_exchange_share'], 1.0-row['off_exchange_share']])
        plt.title('Estimated off-exchange share (sample)')
        plt.ylabel('share')
        plt.tight_layout()
        plt.show()
    else:
        print('No quantitative plot: off-exchange check is not applicable without venue mapping.')

if plt is not None and gran_file.height > 0:
    gp = pd.DataFrame(gran_file.select(['off_exchange_share']).to_dicts())
    if not gp.empty:
        plt.figure(figsize=(8,4))
        plt.hist(gp['off_exchange_share'], bins=40)
        plt.title('Off-exchange share by file distribution')
        plt.xlabel('off_exchange_share')
        plt.ylabel('count')
        plt.tight_layout()
        plt.show()



No quantitative plot: off-exchange check is not applicable without venue mapping.


**Interpretacion de salida anterior**

Si no hay gr?fica cuantitativa, el control qued? `NOT_APPLICABLE` por falta de detectabilidad de venue.


## Paso 4 - Calibracion y decision


In [6]:
runs_root = PROJECT_ROOT / 'runs' / 'data_quality' / NOTEBOOK_ID


def _quantile(vals, q):
    if not vals:
        return float('nan')
    x=sorted(float(v) for v in vals)
    if len(x)==1:
        return x[0]
    pos=(len(x)-1)*q
    lo=int(pos)
    hi=min(lo+1,len(x)-1)
    frac=pos-lo
    return x[lo]*(1-frac)+x[hi]*frac


def _calibrate(vals, default_pass, default_warn):
    vals=[float(v) for v in vals if v is not None]
    n=len(vals)
    if n>=20:
        p_pass=_quantile(vals,0.95); p_warn=_quantile(vals,0.99); mode='hist_p95_p99'
    elif n>=5:
        p_pass=_quantile(vals,0.90); p_warn=_quantile(vals,0.98); mode='hist_p90_p98'
    elif n>0:
        p_pass=max(vals); p_warn=max(vals)*1.25; mode='hist_max_buffer'
    else:
        p_pass=default_pass; p_warn=default_warn; mode='default_fallback'
    p_pass=max(0.0,float(p_pass)); p_warn=max(p_pass+1e-9,float(p_warn))
    return {'n_history':n,'mode':mode,'pass_max':p_pass,'warn_max':p_warn}


def _status(v,pass_max,warn_max):
    if v is None:
        return 'WARN','metric_not_available'
    v=float(v)
    if v<=pass_max: return 'PASS','within_pass_threshold'
    if v<=warn_max: return 'WARN','within_warn_threshold'
    return 'FAIL','exceeds_warn_threshold'

metrics = schema_gate.hstack(offex.select(['n_rows','total_qty','off_exchange_qty','off_exchange_share','locked_applicability']))
mrow=metrics.to_dicts()[0] if metrics.height>0 else {}

if mrow.get('off_exchange_check_status') != 'available':
    applicability_status='NOT_APPLICABLE'
    gate_breakdown=[
        {'gate':'venue_detectability','status':'NOT_APPLICABLE','reason':mrow.get('venue_detectability')},
        {'gate':'off_exchange_share','status':'NOT_APPLICABLE','reason':'metric_not_reliable_without_mapping','value':None},
    ]
    calibration={'off_exchange_share':{'n_history':0,'mode':'disabled','pass_max':None,'warn_max':None}}
    overall_status='NOT_APPLICABLE'
else:
    hist=[]
    if runs_root.exists():
        for d in sorted([x for x in runs_root.iterdir() if x.is_dir()], key=lambda x:x.stat().st_mtime):
            if d.name==RUN_ID: continue
            mp=d/'off_exchange_metrics.parquet'
            if mp.exists():
                try:
                    hm=pl.read_parquet(mp)
                    if hm.height>0:
                        v=hm.to_dicts()[0].get('off_exchange_share')
                        if v is not None: hist.append(float(v))
                except Exception:
                    pass
    thr=_calibrate(hist, default_pass=0.45, default_warn=0.65)
    st,rs=_status(mrow.get('off_exchange_share'),thr['pass_max'],thr['warn_max'])
    gate_breakdown=[
        {'gate':'venue_detectability','status':'PASS','reason':mrow.get('venue_detectability')},
        {'gate':'off_exchange_share','status':st,'reason':rs,'value':mrow.get('off_exchange_share'),'pass_max':thr['pass_max'],'warn_max':thr['warn_max'],'history_n':thr['n_history'],'mode':thr['mode']},
    ]
    calibration={'off_exchange_share':thr}
    overall_status='FAIL' if st=='FAIL' else ('WARN' if st=='WARN' else 'PASS')

recommended_actions=[]
if applicability_status=='NOT_APPLICABLE':
    recommended_actions.append({'priority':'high','action':'venue_mapping_required','detail':'Definir mapeo robusto de c?digos de venue/TRF antes de usar off-exchange share.'})
elif overall_status in {'WARN','FAIL'}:
    recommended_actions.append({'priority':'high' if overall_status=='FAIL' else 'medium','action':'investigate_off_exchange_concentration','detail':'Analizar concentraci?n por day/session/venue y contrastar con referencia externa FINRA.'})

print('Applicability:', applicability_status)
print('Gate breakdown:', gate_breakdown)
print('OVERALL_STATUS:', overall_status)
print('Recommended actions:', recommended_actions)



Applicability: NOT_APPLICABLE
Gate breakdown: [{'gate': 'venue_detectability', 'status': 'NOT_APPLICABLE', 'reason': 'low_confidence_unmapped_codes'}, {'gate': 'off_exchange_share', 'status': 'NOT_APPLICABLE', 'reason': 'metric_not_reliable_without_mapping', 'value': None}]
OVERALL_STATUS: NOT_APPLICABLE
Recommended actions: [{'priority': 'high', 'action': 'venue_mapping_required', 'detail': 'Definir mapeo robusto de c?digos de venue/TRF antes de usar off-exchange share.'}]


**Interpretacion de salida anterior**

Este bloque separa dos casos: medici?n defendible (`APPLICABLE`) vs ausencia de mapeo (`NOT_APPLICABLE`).


## Paso 5 - Export y decision


In [7]:
if metrics.height > 0:
    metrics.write_parquet(OUT_DIR / 'off_exchange_metrics.parquet')
if gran_file.height > 0:
    gran_file.write_parquet(OUT_DIR / 'off_exchange_granularity_by_file.parquet')
if venue_profile.height > 0:
    venue_profile.write_parquet(OUT_DIR / 'off_exchange_venue_profile.parquet')

with open(OUT_DIR / 'off_exchange_calibration.json', 'w', encoding='utf-8') as f:
    json.dump(calibration, f, indent=2)

statuses = [g.get('status', 'PASS') for g in gate_breakdown]
if applicability_status == 'NOT_APPLICABLE':
    root_cause = 'schema_gap'
elif any(s in {'FAIL','WARN'} for s in statuses):
    root_cause = 'microstructure_friction'
else:
    root_cause = 'none'

decision_table = [{
    'ticker': SYMBOLS[0] if SYMBOLS else '<unknown>',
    'applicability_status': applicability_status,
    'overall_status': overall_status,
    'root_cause': root_cause,
    'decision': overall_status,
}]

decision={
    'check_id':'off_exchange_trf_reconciliation_v2',
    'as_of_utc':datetime.now(timezone.utc).isoformat(),
    'git_commit':git_commit,
    'symbols':SYMBOLS,
    'applicability_status':applicability_status,
    'overall_status':overall_status,
    'root_cause': root_cause,
    'metrics':metrics.to_dicts(),
    'gate_breakdown':gate_breakdown,
    'decision_table': decision_table,
    'calibration':calibration,
    'recommended_actions':recommended_actions,
    'scope_note':'Result is reliable only when venue mapping to TRF/off-exchange is defensible.',
}
with open(OUT_DIR / 'off_exchange_decision.json','w',encoding='utf-8') as f:
    json.dump(decision,f,indent=2)

print('Saved:', OUT_DIR / 'off_exchange_metrics.parquet')
print('Saved:', OUT_DIR / 'off_exchange_granularity_by_file.parquet')
print('Saved:', OUT_DIR / 'off_exchange_venue_profile.parquet')
print('Saved:', OUT_DIR / 'off_exchange_calibration.json')
print('Saved:', OUT_DIR / 'off_exchange_decision.json')
print('APPLICABILITY_STATUS:', applicability_status)
print('OVERALL_STATUS:', overall_status)



Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\12_off_exchange_trf_reconciliation\20260213_183454_19355fad\off_exchange_metrics.parquet
Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\12_off_exchange_trf_reconciliation\20260213_183454_19355fad\off_exchange_granularity_by_file.parquet
Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\12_off_exchange_trf_reconciliation\20260213_183454_19355fad\off_exchange_venue_profile.parquet
Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\12_off_exchange_trf_reconciliation\20260213_183454_19355fad\off_exchange_calibration.json
Saved: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\12_off_exchange_trf_reconciliation\20260213_183454_19355fad\off_exchange_decision.json
APPLICABILITY_STATUS: NOT_APPLICABLE
OVERALL_STATUS: NOT_APPLICABLE


**Interpretacion final**

El objetivo no es forzar un n?mero, sino distinguir cu?ndo la estimaci?n de off-exchange es t?cnicamente defendible.
